# Gloss-Free Sign Language Translation (ASL-to-English)
## Portfolio Training Notebook for Kaggle GPUs

This notebook provides a complete pipeline to clone the landmark-based ASL translation repository, set up the dependency environment, validate MediaPipe Holistic datasets, run sanitization unit tests, train our hybrid **Conformer-T5 model** with **Multi-Stream Gated Fusion** (fusing 3D landmarks and precomputed I3D features), and export the trained model to ONNX for fast inference.

## 1. Setup, Environment Validation, and Sanity Checks

In [ ]:
# Clone or update the repository
import os
if not os.path.exists('/kaggle/working/gloss-free-asl-translation'):
    !git clone https://github.com/yyouretoast/gloss-free-asl-translation.git
    %cd /kaggle/working/gloss-free-asl-translation
else:
    %cd /kaggle/working/gloss-free-asl-translation
    !git checkout -- requirements.txt
    !git pull


In [ ]:
# Filter out torch/torchvision to keep Kaggle's GPU-optimized pre-installs
!sed -i '/torch/d' requirements.txt
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Run environment validation to verify GPU availability and dependencies
!python -m scripts.check_environment


## 1. Setup, Environment Validation, and Sanity Checks

In [ ]:
# Run unit tests (excluding slow dataset integration tests)
!pytest tests/ -v -m "not slow"


## 2. Environment Configuration (Optional HF Token)

In [ ]:
import os
# Set your HF Token if available
# os.environ["HF_TOKEN"] = "your_huggingface_token_here"

## 3. Real-World Dataset Profiling & Validation

In [ ]:
# Run dataset validation profile on a subset of files to verify shapes and integrity
import os
import glob

# Auto-resolve How2Sign Holistic dataset paths on Kaggle
candidates = [
    '/kaggle/input/how2sign-holistic',
    '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic',
    '/kaggle/input/how2sign-holistic/how2sign_holistic'
]
data_dir = next((p for p in candidates if os.path.exists(p)), None)
if data_dir:
    # If path directly contains .npy files, validate it. Otherwise, look for subdirs
    npy_files = glob.glob(os.path.join(data_dir, '*.npy'))
    if not npy_files:
        subdirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        if subdirs:
            data_dir = subdirs[0]
    print(f"Target data directory resolved: {data_dir}")
    !python -m src.validate_dataset --data_dir {data_dir} --limit 50
else:
    print("How2Sign Holistic dataset not found! Please attach 'how2sign-holistic' to your Kaggle notebook.")


In [ ]:
# Inspect the shapes and channels of the attached Holistic landmarks and I3D features
import os
import glob
import numpy as np

# 1. Audit Landmarks
candidates_lm = [
    '/kaggle/input/how2sign-holistic',
    '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic'
]
lm_dir = next((p for p in candidates_lm if os.path.exists(p)), None)
if lm_dir:
    # List files directly in the directory instead of recursive globbing (which is slow)
    if os.path.isdir(os.path.join(lm_dir, 'how2sign_holistic')):
        lm_dir = os.path.join(lm_dir, 'how2sign_holistic')
    npy_files = sorted([os.path.join(lm_dir, f) for f in os.listdir(lm_dir) if f.endswith('.npy')])
    if npy_files:
        print(f"Found {len(npy_files)} landmark files in {lm_dir}.")
        test_file = npy_files[0]
        print(f"Auditing landmarks file '{os.path.basename(test_file)}':")
        data = np.load(test_file)
        print(f" - Array Shape: {data.shape} (Expected: (num_frames, 543, 3))")

# 2. Audit I3D Features
candidates_i3d = [
    '/kaggle/input/how2sign-i3d-features',
    '/kaggle/input/datasets/muhammadzakriya/how2sign-i3d-features'
]
i3d_dir = next((p for p in candidates_i3d if os.path.exists(p)), None)
if i3d_dir:
    # Resolve internal directory if nested
    if os.path.isdir(os.path.join(i3d_dir, 'how2sign_i3d_features')):
        i3d_dir = os.path.join(i3d_dir, 'how2sign_i3d_features')
    i3d_files = sorted([os.path.join(i3d_dir, f) for f in os.listdir(i3d_dir) if f.endswith('.npy')])
    if i3d_files:
        print(f"Found {len(i3d_files)} I3D feature files in {i3d_dir}.")
        test_i3d = i3d_files[0]
        print(f"Auditing I3D file '{os.path.basename(test_i3d)}':")
        data_i3d = np.load(test_i3d)
        print(f" - Feature Shape: {data_i3d.shape} (Expected: (num_frames, 1024))")


### 4.2 Persist Preprocessed Dataset for Future Runs (Optional)

Since preprocessing the 31,000+ folders of How2Sign takes approximately 2 hours, zipping the preprocessed `.npz` files and saving them to Kaggle's `/kaggle/working` directory allows you to download them or persist them as a Kaggle output dataset. This lets you skip the preprocessing phase entirely in future runs.

## 4. End-to-End Conformer-T5 Multi-Stream Model Training

This runs our end-to-end `Conformer -> T5-Small` translation pipeline using **Multi-Stream Gated Fusion** to combine skeletal trajectories and precomputed I3D spatiotemporal visual features. Adjust batch size and epochs as needed.

In [ ]:
# --- Option A: Load TensorBoard visualization ---
%load_ext tensorboard
%tensorboard --logdir results/checkpoints/runs


In [ ]:
# --- Option A: How2Sign Multi-Stream Training ---
import os
import glob

# 1. Resolve Landmark Directory
candidates_lm = [
    '/kaggle/input/how2sign-holistic',
    '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic'
]
lm_dir = next((p for p in candidates_lm if os.path.exists(p)), None)
if lm_dir:
    # Find the actual directory containing npy files
    npy_files = glob.glob(os.path.join(lm_dir, '*.npy'))
    if not npy_files:
        subdirs = [os.path.join(lm_dir, d) for d in os.listdir(lm_dir) if os.path.isdir(os.path.join(lm_dir, d))]
        if subdirs:
            lm_dir = subdirs[0]

# 2. Resolve I3D Directory
candidates_i3d = [
    '/kaggle/input/how2sign-i3d-features',
    '/kaggle/input/datasets/muhammadzakriya/how2sign-i3d-features'
]
i3d_dir = next((p for p in candidates_i3d if os.path.exists(p)), None)
if i3d_dir:
    npy_files = glob.glob(os.path.join(i3d_dir, '*.npy'))
    if not npy_files:
        subdirs = [os.path.join(i3d_dir, d) for d in os.listdir(i3d_dir) if os.path.isdir(os.path.join(i3d_dir, d))]
        if subdirs:
            i3d_dir = subdirs[0]

# 3. Resolve Metadata CSV file (train split)
candidates_csv = [
    '/kaggle/input/how2sign-holistic/how2sign_realigned_train.csv',
    '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic/how2sign_realigned_train.csv',
    '/kaggle/input/how2sign-keypoints/how2sign_realigned_train.csv',
    '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv'
]
metadata_file = next((p for p in candidates_csv if os.path.exists(p)), None)

print(f"Landmark Directory: {lm_dir}")
print(f"I3D Directory:      {i3d_dir}")
print(f"Metadata File:      {metadata_file}")

if lm_dir and i3d_dir and metadata_file:
    # Run training with I3D multi-stream fusion enabled
    !python -m src.train --epochs 50 --batch_size 8 --lr 1e-4 --data_dir {lm_dir} --i3d_dir {i3d_dir} --metadata_file {metadata_file} --resume_from_checkpoint latest
else:
    print("Error: Missing directories. Please verify how2sign-holistic and how2sign-i3d-features are attached.")


## 5. Model Optimization & Export (Conformer Encoder to ONNX)

In [ ]:
# Trace and export the best checkpoint to ONNX
import os
import glob

checkpoint_dirs = sorted(glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]
    model_bin = os.path.join(best_checkpoint, "model.safetensors")
    if not os.path.exists(model_bin):
        model_bin = os.path.join(best_checkpoint, "pytorch_model.bin")
    
    # 501 dimensions for face-enabled How2Sign Holistic (.npy), 225 for face-disabled ablation run
    input_dim = 501  # CHANGE this value to 225 if you ran training with --no_face!
    
    print(f"Exporting encoder from {model_bin} to ONNX with input_dim={input_dim}...")
    !python -X utf8 scripts/export_onnx.py --input-dim {input_dim} --model-path {model_bin} --output /kaggle/working/conformer_encoder.onnx
else:
    print("No checkpoints found to export!")


## 6. Checkpoint Archival & Retrieval

In [ ]:
# Zip all checkpoints for easy download
!zip -q -r /kaggle/working/checkpoints.zip results/checkpoints
print("Zipped checkpoints to /kaggle/working/checkpoints.zip")

## 7. Quantitative Evaluation & Qualitative Inference

In [ ]:
# --- Run Quantitative Evaluation ---
import glob
import os

checkpoint_dirs = sorted(glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]
    
    # Resolve directories dynamically (same as training cell)
    candidates_lm = [
        '/kaggle/input/how2sign-holistic',
        '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic'
    ]
    lm_dir = next((p for p in candidates_lm if os.path.exists(p)), None)
    if lm_dir:
        if os.path.isdir(os.path.join(lm_dir, 'how2sign_holistic')):
            lm_dir = os.path.join(lm_dir, 'how2sign_holistic')
            
    candidates_i3d = [
        '/kaggle/input/how2sign-i3d-features',
        '/kaggle/input/datasets/muhammadzakriya/how2sign-i3d-features'
    ]
    i3d_dir = next((p for p in candidates_i3d if os.path.exists(p)), None)
    if i3d_dir:
        if os.path.isdir(os.path.join(i3d_dir, 'how2sign_i3d_features')):
            i3d_dir = os.path.join(i3d_dir, 'how2sign_i3d_features')
            
    candidates_csv = [
        '/kaggle/input/how2sign-holistic/how2sign_realigned_train.csv',
        '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic/how2sign_realigned_train.csv',
        '/kaggle/input/how2sign-keypoints/how2sign_realigned_train.csv',
        '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv'
    ]
    metadata_file = next((p for p in candidates_csv if os.path.exists(p)), None)
    
    !pip install -q jiwer sacrebleu
    
    print(f"Evaluating best checkpoint: {best_checkpoint}")
    print(f" - Landmarks: {lm_dir}")
    print(f" - I3D:       {i3d_dir}")
    print(f" - Metadata:  {metadata_file}")
    
    # Pass explicit t5-small model parameter
    !python -m scripts.evaluate --checkpoint {best_checkpoint} --data-dir {lm_dir} --i3d-dir {i3d_dir} --metadata {metadata_file} --t5-model t5-small
else:
    print("No checkpoints found to evaluate!")


## 7. Quantitative Evaluation & Qualitative Inference

In [ ]:
# --- Run Qualitative Sample Inference ---
import os
import glob
import torch
import numpy as np
from transformers import T5TokenizerFast
from src.models.translation_model import ASLTranslationModel
from src.dataset import ASLLandmarkDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_dirs = sorted(glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]
    model_path = os.path.join(best_checkpoint, "model.safetensors")
    if not os.path.exists(model_path):
        model_path = os.path.join(best_checkpoint, "pytorch_model.bin")
    
    # Resolve directories dynamically
    candidates_lm = [
        '/kaggle/input/how2sign-holistic',
        '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic'
    ]
    lm_dir = next((p for p in candidates_lm if os.path.exists(p)), None)
    if lm_dir:
        if os.path.isdir(os.path.join(lm_dir, 'how2sign_holistic')):
            lm_dir = os.path.join(lm_dir, 'how2sign_holistic')
            
    candidates_i3d = [
        '/kaggle/input/how2sign-i3d-features',
        '/kaggle/input/datasets/muhammadzakriya/how2sign-i3d-features'
    ]
    i3d_dir = next((p for p in candidates_i3d if os.path.exists(p)), None)
    if i3d_dir:
        if os.path.isdir(os.path.join(i3d_dir, 'how2sign_i3d_features')):
            i3d_dir = os.path.join(i3d_dir, 'how2sign_i3d_features')
            
    candidates_csv = [
        '/kaggle/input/how2sign-holistic/how2sign_realigned_train.csv',
        '/kaggle/input/datasets/pasindusewmuthuabewickramasinghe/how2sign-holistic/how2sign_realigned_train.csv',
        '/kaggle/input/how2sign-keypoints/how2sign_realigned_train.csv',
        '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv'
    ]
    metadata_file = next((p for p in candidates_csv if os.path.exists(p)), None)
    
    print(f"Loading best checkpoint from {model_path}...")
    
    # Load metadata
    from src.utils.metadata import load_metadata
    metadata, _ = load_metadata(metadata_file)
    
    # Setup tokenizer with t5-small
    tokenizer = T5TokenizerFast.from_pretrained("t5-small")
    dataset = ASLLandmarkDataset(
        data_dir=lm_dir,
        metadata_dict=metadata,
        max_len=150,
        include_face=True,
        normalize=True,
        skip_empty_labels=True,
        i3d_dir=i3d_dir
    )
    
    if len(dataset) > 0:
        input_dim = dataset[0]['features'].shape[1]
        # Auto-detect gating weights
        if model_path.endswith(".safetensors"):
            from safetensors.torch import load_file
            state_dict = load_file(model_path, device="cpu")
        else:
            state_dict = torch.load(model_path, map_location="cpu", weights_only=True)
            
        # Strip prefix
        state_dict = {k[7:] if k.startswith("module.") else k: v for k, v in state_dict.items()}
        has_i3d_weights = any('i3d_projection' in k or 'gate_conv' in k for k in state_dict.keys())
        input_i3d_dim = 1024 if has_i3d_weights else None
        
        model = ASLTranslationModel(
            input_dim=input_dim,
            input_i3d_dim=input_i3d_dim,
            d_model=512,
            t5_model_name="t5-small",
            num_layers=4,
            num_heads=4,
            kernel_size=31
        )
        model.load_state_dict(state_dict)
        model = model.to(device)
        model.eval()
        
        import pandas as pd
        print("\n--- Qualitative Predictions ---")
        num_samples = min(5, len(dataset))
        targets = []
        predictions = []
        for i in range(num_samples):
            sample = dataset[i]
            features = sample['features'].unsqueeze(0).to(device)
            attention_mask = torch.ones((1, features.shape[1]), dtype=torch.float32, device=device)
            i3d_feats = sample['i3d_features'].unsqueeze(0).to(device) if 'i3d_features' in sample else None
            
            with torch.no_grad():
                output_ids = model.generate(
                    input_features=features,
                    attention_mask=attention_mask,
                    input_i3d_features=i3d_feats,
                    max_new_tokens=30,
                    num_beams=5 # Pass default beam search to match eval
                )
            pred_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            targets.append(sample['text'])
            predictions.append(pred_text)
            
        results_df = pd.DataFrame({
            "Sample": [f"#{i+1}" for i in range(num_samples)],
            "Target (Ground Truth)": targets,
            "Predicted Translation": predictions
        })
        from IPython.display import display, HTML
        display(HTML(results_df.to_html(index=False)))
else:
    print("No checkpoints found for qualitative inference!")
